# Lab 5 - Quiz Generator (NotebookLM-style), without reading the whole PDF at once

**Goal:** generate a multiple-choice quiz from a PDF that is **too big to fit in one
prompt** - so we never send the whole document to the LLM.

**The pattern: map -> reduce** (a plain Python loop, LangChain for the pieces):

```
              batch 1 of chunks --> LLM --> 2 questions --+
PDF -> chunks  batch 2 of chunks --> LLM --> 2 questions --+--> question bank --> LLM (questions only) --> cleaned quiz
              batch 3 of chunks --> LLM --> 2 questions --+        (dedupe, pick best N)
                     ...
```

- **map:** loop the PDF in small batches, ask for a couple of questions grounded *only* in
  that batch. Each call is tiny, so the token limit is never a problem, and every section
  of the document gets covered.
- **reduce:** send just the collected **questions** (not the document) back once to remove
  duplicates and keep the best ones. Questions are short, so this always fits.

> Why not RAG here? Retrieval only pulls chunks similar to a query, so whole sections of
> the document would never be quizzed. RAG is the right tool when you want a quiz on **one
> named topic** - that variant is in the exercises.

| Piece | Choice |
|-------|--------|
| Loader / splitter | `PyPDFLoader` + `RecursiveCharacterTextSplitter` |
| LLM | Groq `openai/gpt-oss-120b` |
| Quiz UI | Gradio radio buttons + scoring |

No embeddings or vector store in this lab - it is pure map-reduce.


## Step 0 - Install

In [4]:
%pip install -q langchain langchain-community langchain-groq \ langchain-text-splitters pypdf gradio

## Step 1 - Groq API key

Free key: https://console.groq.com/keys . Colab secret name: `GROQ_API_KEY`.

In [5]:
import os

def load_key(name: str) -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: loaded from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: loaded from environment"); return os.environ[name]
    from getpass import getpass
    return getpass(f"Paste your {name}: ")

GROQ_API_KEY = load_key("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("Key ends with:", GROQ_API_KEY[-4:])

GROQ_API_KEY: loaded from Colab secret
Key ends with: kRn5


## Step 2 - Choose a PDF

Any of the PDFs from `data/`, or upload your own. `minion-tech.pdf` (a 22-page business
document) is the default because it has the most varied content to quiz on.

In [7]:
PDF_NAME = "minion-tech.pdf"   # or "PlacementGuide.pdf" or "company-sustainability-report-2022.pdf"

def get_pdf(name):
    for path in (name, f"data/{name}", f"RAG_Labs/data/{name}"):
        if os.path.exists(path):
            print("Found:", path); return path
    try:
        from google.colab import files
        print(f"Choose {name} (or any PDF)..."); up = files.upload()
        return list(up.keys())[0]
    except Exception:
        raise FileNotFoundError(name)

PDF_PATH = get_pdf(PDF_NAME)

Choose minion-tech.pdf (or any PDF)...


Saving PlacementGuide.pdf to PlacementGuide (1).pdf


## Step 3 - Load and chunk the PDF

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(PDF_PATH).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150).split_documents(pages)
print(f"{len(pages)} pages -> {len(chunks)} chunks (~1200 chars each)")

/tmp/ipykernel_842/2976734443.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


3 pages -> 14 chunks (~1200 chars each)


## Step 4 - Why we loop instead of sending the whole document

Rule of thumb: **1 token ~ 4 characters**. The sample PDF is small, but the point is what
happens as the document grows - and generating one question per prompt from a huge context
is wasteful and eventually will not fit.

In [10]:
total_chars = sum(len(c.page_content) for c in chunks)
approx_tokens = total_chars // 4
per_batch_tokens = 4 * 1200 // 4

print(f"This PDF           : ~{approx_tokens:,} tokens ({total_chars:,} chars)")
print(f"One batch of 4 chunks : ~{per_batch_tokens:,} tokens  <- what we actually send per call")
print()
print("Scale it up (same ~4 chars/token):")
for pages_n, label in [(50, "a 50-page report"), (300, "a 300-page textbook")]:
    est = approx_tokens * pages_n / max(len(pages), 1)
    print(f"  {label:22s} ~ {est:,.0f} tokens  -> will not fit in one prompt")
print("\nSo: loop the document in small, fixed-size batches. Cost stays predictable.")

This PDF           : ~3,723 tokens (14,894 chars)
One batch of 4 chunks : ~1,200 tokens  <- what we actually send per call

Scale it up (same ~4 chars/token):
  a 50-page report       ~ 62,050 tokens  -> will not fit in one prompt
  a 300-page textbook    ~ 372,300 tokens  -> will not fit in one prompt

So: loop the document in small, fixed-size batches. Cost stays predictable.


## Step 5 - Skip the boilerplate chunks

Title pages, tables of contents and number-only chunks make bad questions. A cheap filter:
drop chunks that are very short or mostly digits/punctuation.

In [11]:
def is_quizzable(text: str) -> bool:
    t = text.strip()
    if len(t) < 300:
        return False
    letters = sum(ch.isalpha() for ch in t)
    return letters / len(t) > 0.55   # at least 55% actual letters

good_chunks = [c for c in chunks if is_quizzable(c.page_content)]
print(f"{len(chunks)} chunks -> {len(good_chunks)} worth quizzing "
      f"({len(chunks) - len(good_chunks)} skipped as boilerplate)")

14 chunks -> 13 worth quizzing (1 skipped as boilerplate)


## Step 6 - The LLM

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, temperature=0.3)
print(llm.invoke("Reply with one word: ready").content)

ready


## Step 7 - MAP: loop the document, a few chunks at a time

For each batch of `BATCH` chunks we ask for `PER_BATCH` questions, grounded **only** in that
batch's text. We print every question as it is generated so you can watch the bank fill up.

Each question is JSON:
```
{"question": "...", "options": ["A","B","C","D"], "answer_index": 0,
 "explanation": "...", "page": 7}
```

In [13]:
import json, re, time

BATCH = 4          # chunks per LLM call  (keeps each call small)
PER_BATCH = 3      # questions asked per call

MAP_PROMPT = """You are writing a multiple-choice quiz from a document.
Using ONLY the text below, write exactly {n} good quiz questions.

Rules:
- 4 options each, exactly one correct.
- Test understanding of a fact stated in the text, not trivia about formatting.
- "answer_index" is the 0-based index of the correct option.
- "page" is the page number the answer comes from (given in brackets).
- "explanation" is one sentence quoting/paraphrasing the supporting text.

TEXT:
{context}

Reply with ONLY a JSON list of {n} objects, each:
{{"question": str, "options": [str, str, str, str], "answer_index": int, "explanation": str, "page": int}}"""

def parse_json_list(raw: str):
    cleaned = re.sub(r"```(json)?", "", raw).strip()
    m = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if m:
        cleaned = m.group(0)
    try:
        return json.loads(cleaned)
    except Exception:
        return []

def llm_json(prompt, tries=3):
    for i in range(tries):
        raw = llm.invoke(prompt).content
        items = parse_json_list(raw)
        if items:
            return items
        print(f"   (parse retry {i + 1}/{tries})"); time.sleep(1)
    return []

def build_question_bank(source_chunks, batch=BATCH, per_batch=PER_BATCH, verbose=True):
    bank = []
    for start in range(0, len(source_chunks), batch):
        group = source_chunks[start:start + batch]
        context = "\n\n".join(f"[page {c.metadata.get('page', '?')}] {c.page_content}" for c in group)
        prompt = MAP_PROMPT.format(n=per_batch, context=context)

        items = llm_json(prompt)
        batch_no = start // batch + 1
        for it in items:
            if isinstance(it, dict) and it.get("question") and len(it.get("options", [])) == 4:
                it["batch"] = batch_no
                bank.append(it)
                if verbose:
                    print(f"  batch {batch_no:2d}: {it['question'][:90]}")
    return bank

question_bank = build_question_bank(good_chunks)
print(f"\nMAP done: {len(question_bank)} raw questions from {len(good_chunks)} chunks")

  batch  1: According to the text, what is the most important factor that determines who gets hired?
  batch  1: Which behavior is specifically advised against as part of good body language during an int
  batch  1: When an interviewer asks unexpected or provocative questions, how should a candidate respo
  batch  2: What trait is identified as a key to professional and organizational growth?
  batch  2: Which of the following is given as an example of a bad resume title?
  batch  2: According to the resume guidelines, which item should not be included unless specifically 
  batch  3: In the Personal Details section of a good resume, which information is identified as the m
  batch  3: According to the passage, why is a Group Discussion included in the selection process?
  batch  3: Which of the following examples is listed as a "Factual topic" for a Group Discussion?
  batch  4: What is the objective of the case study as described in the text?
  batch  4: Which of the following topics

## Step 8 - REDUCE: clean the bank (questions only - no document)

Now we send just the **questions** back to the LLM (short - always fits) and ask it to
drop duplicates and near-duplicates, remove anything ambiguous or answerable without the
document, and keep the best `TARGET` spread across the whole document.

In [14]:
TARGET = 20

REDUCE_PROMPT = """Here is a raw bank of quiz questions built from different parts of one document.

Clean it:
- remove exact duplicates and questions that test the same fact
- remove anything ambiguous, opinion-based, or answerable without the document
- keep the {target} strongest questions, spread across as many different "batch" numbers as possible
- keep each kept question EXACTLY as-is (same options, same answer_index, same page)

RAW BANK (JSON):
{bank}

Reply with ONLY a JSON list of the {target} kept question objects (same shape as the input, you may drop the "batch" field)."""

def curate(bank, target=TARGET, verbose=True):
    target = min(target, len(bank))          # can't keep more than we generated
    prompt = REDUCE_PROMPT.format(target=target, bank=json.dumps(bank, ensure_ascii=False))
    kept = llm_json(prompt)
    # safety net: if the reduce step misbehaves, fall back to a simple spread
    if not (target - 2 <= len(kept) <= target + 4):
        if verbose:
            print("   (reduce step off-target, using a simple fallback selection)")
        seen, kept = set(), []
        for q in bank:
            key = q["question"][:40].lower()
            if key not in seen:
                seen.add(key); kept.append(q)
            if len(kept) >= target:
                break
    # final validation
    clean = []
    for q in kept:
        if (q.get("question") and len(q.get("options", [])) == 4
                and isinstance(q.get("answer_index"), int) and 0 <= q["answer_index"] <= 3):
            q.pop("batch", None)
            clean.append(q)
    return clean

quiz = curate(question_bank)
print(f"REDUCE done: {len(question_bank)} raw -> {len(quiz)} final questions\n")
for i, q in enumerate(quiz, 1):
    print(f"Q{i}. {q['question']}")
    for j, opt in enumerate(q["options"]):
        mark = "*" if j == q["answer_index"] else " "
        print(f"   {mark} {chr(65 + j)}. {opt}")
    print(f"   -> {q['explanation']} (page {q.get('page', '?')})\n")

REDUCE done: 12 raw -> 12 final questions

Q1. According to the text, what is the most important factor that determines who gets hired?
   * A. Attitude
     B. Technical qualifications
     C. Years of experience
     D. Resume format
   -> The text states that the most important factor is attitude, not qualifications, experience, or the resume. (page 0)

Q2. Which behavior is specifically advised against as part of good body language during an interview?
     A. Maintaining eye contact
   * B. Yawning
     C. Giving a firm handshake
     D. Keeping an upright posture
   -> The text lists actions to avoid, including yawning, when describing proper body language. (page 0)

Q3. When an interviewer asks unexpected or provocative questions, how should a candidate respond?
     A. Get angry and confront the interviewer
   * B. Answer honestly and calmly without offending the interviewer
     C. Agree with everything the interviewer says
     D. Change the subject
   -> The text advises sta

## Step 9 - A quick automatic check of the quiz

In [15]:
problems = 0
for i, q in enumerate(quiz, 1):
    if len(set(o.strip().lower() for o in q["options"])) != 4:
        print(f"Q{i}: duplicate options"); problems += 1
    if not (0 <= q["answer_index"] <= 3):
        print(f"Q{i}: bad answer_index"); problems += 1
    if len(q["question"]) < 15:
        print(f"Q{i}: suspiciously short question"); problems += 1
print("All questions structurally OK" if problems == 0 else f"{problems} issue(s) above")

All questions structurally OK


## Step 10 - Gradio quiz app

Take the quiz, submit, get a score and a per-question explanation. "New selection"
re-samples from the bank; "Rebuild bank" re-runs the whole map-reduce on the PDF.

In [16]:
import gradio as gr
import random

MAX_Q = 15   # max radio widgets to pre-create
STATE = {"bank": question_bank, "quiz": quiz}

def sample_quiz(n=TARGET):
    """Cheap resample - just pick from the already-built bank, no LLM call."""
    bank = STATE["bank"]
    picked = random.sample(bank, min(n, len(bank)))
    STATE["quiz"] = picked
    return picked

def render(quiz):
    updates = []
    for i in range(MAX_Q):
        if i < len(quiz):
            q = quiz[i]
            labels = [f"{chr(65 + j)}. {opt}" for j, opt in enumerate(q["options"])]
            updates.append(gr.update(label=f"Q{i + 1}. {q['question']}", choices=labels,
                                     value=None, visible=True))
        else:
            updates.append(gr.update(visible=False))
    return updates

def grade(*answers):
    quiz = STATE["quiz"]
    score = 0
    lines = []
    for i, q in enumerate(quiz):
        chosen = answers[i]
        chosen_idx = (ord(chosen[0]) - 65) if chosen else -1
        ok = chosen_idx == q["answer_index"]
        score += ok
        correct = f"{chr(65 + q['answer_index'])}. {q['options'][q['answer_index']]}"
        lines.append(f"**Q{i + 1}. {q['question']}**\n"
                     f"- Your answer: {chosen or '(none)'}  {'CORRECT' if ok else 'WRONG'}\n"
                     f"- Correct: {correct}\n"
                     f"- {q['explanation']}  _(page {q.get('page', '?')})_")
    header = f"## Score: {score} / {len(quiz)}\n\n"
    return header + "\n\n".join(lines)

with gr.Blocks(title="Lab 5 - Quiz Generator") as demo:
    gr.Markdown(f"# Lab 5 - Quiz from **{PDF_NAME}**\nGenerated by map-reduce over the PDF - the whole document was never sent at once.")
    radios = [gr.Radio(choices=[], label=f"Q{i + 1}", visible=False) for i in range(MAX_Q)]
    with gr.Row():
        submit = gr.Button("Submit answers", variant="primary")
        resample = gr.Button("New selection")
        rebuild = gr.Button("Rebuild bank from PDF (slow)")
    result = gr.Markdown()

    submit.click(grade, inputs=radios, outputs=result)
    resample.click(lambda: render(sample_quiz()), outputs=radios)

    def do_rebuild():
        STATE["bank"] = build_question_bank(good_chunks, verbose=False)
        STATE["quiz"] = curate(STATE["bank"], verbose=False)
        return render(STATE["quiz"])
    rebuild.click(do_rebuild, outputs=radios)

    demo.load(lambda: render(STATE["quiz"]), outputs=radios)

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1941d691d6d3282ace.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Recap

- **Never sent the whole PDF to the LLM.** The map step processed 4 chunks at a time; the
  reduce step saw only the generated questions.
- **map** gives whole-document coverage; **reduce** gives quality (dedupe + pick best).
- Total cost = (chunks / batch) small calls + 1 reduce call - predictable and bounded.

### Exercises
1. **Topic-focused quiz with RAG.** Instead of looping every chunk, build a FAISS index
   (Lab 1), retrieve the top ~8 chunks for a topic like *"group discussion"*, and run only
   the map step on those. Compare the questions.
2. Change `PER_BATCH` to 1 and `BATCH` to 2 - more calls, more questions. Does quality go
   up or just quantity?
3. Add a `difficulty` field to the map prompt ("easy"/"medium"/"hard") and let the Gradio
   app filter by it.
4. Swap `PDF_NAME` to `PlacementGuide.pdf` and rebuild - does the boilerplate filter in
   Step 5 still behave well?
